#Feature Engineering


In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

from scipy.sparse import hstack, csr_matrix

import nltk
nltk.download("vader_lexicon")
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import OneHotEncoder

file_path = '/content/drive/MyDrive/FakeReviewsProject/data_for_regression.csv'

df = pd.read_csv(file_path)

print(df.columns)

df["text"] = df["text"].fillna("").astype(str)
df["label"] = df["label"].astype(int)

#Now rating_extremes are 0-2
df["rating_extreme"] = abs(df["rating"] - 3)


#Convert category column into numbers via one hot encoding
ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
category_ohe = ohe.fit_transform(df[["category"]])

#Viewing Data
print(df.columns)
category_ohe.shape
ohe.get_feature_names_out(["category"])[:10]

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Index(['category', 'rating', 'label', 'text'], dtype='object')
Index(['category', 'rating', 'label', 'text', 'rating_extreme'], dtype='object')


array(['category_Books_5', 'category_Clothing_Shoes_and_Jewelry_5',
       'category_Electronics_5', 'category_Home_and_Kitchen_5',
       'category_Kindle_Store_5', 'category_Movies_and_TV_5',
       'category_Pet_Supplies_5', 'category_Sports_and_Outdoors_5',
       'category_Tools_and_Home_Improvement_5',
       'category_Toys_and_Games_5'], dtype=object)

##Lexcial Diversity
How unique the vocab is in a review (Unique words / total words)

Fake reviews often have a lower lexical diversity as they repeat adjectives and are usually very generic.

##Sentiment Extremeness:
Assigns a score (-1,0,1) based on how strong emotion is:
* -1: very negative
* 1: very positive
* 0: neutral

Fake reviews are usually extreme in emotion, so they would have a high absolute value of sentiment extremeness.

In [ ]:
import re
from nltk.sentiment import SentimentIntensityAnalyzer
import numpy as np
from scipy.sparse import csr_matrix
sia = SentimentIntensityAnalyzer()

#extracting word, sentence, exclamation, uppercase ratio, lexical diversity (unique:total ratio), score

def text_numeric_features(text):
    words = text.split()
    sentences = re.split(r"[.!?]+", text)

    word_count = len(words)
    #Dataset was pre-normalized with puncuation and capitalization removed
    #sentence_count = len([s for s in sentences if s.strip()])
    #exclamation_count = text.count("!")
    #uppercase_ratio = sum(c.isupper() for c in text) / len(text) if len(text) > 0 else 0
    lexical_diversity = len(set(words)) / len(words) if len(words) > 0 else 0
    sentiment_extreme = abs(sia.polarity_scores(text)["compound"])

    return [
        word_count,
        #sentence_count,
        #exclamation_count,
        #uppercase_ratio,
        lexical_diversity,
        sentiment_extreme
    ]


text_numeric = np.array([
    text_numeric_features(t) for t in df["text"]
], dtype=float)

text_numeric.shape  # (n_rows, 3)

numeric_feature_names = [
    "word_count",
    #"sentence_count",
    #"exclamation_count",
    #"uppercase_ratio",
    "lexical_diversity",
    "sentiment_extreme"
]

#viewing numeric df head
numeric_df = pd.DataFrame(text_numeric, columns=numeric_feature_names)
numeric_df.head()

,word_count,lexical_diversity,sentiment_extreme
0,8.0,0.875000,0.9517
1,8.0,1.000000,0.8910
2,7.0,0.857143,0.7906
3,6.0,1.000000,0.4404
4,7.0,0.857143,0.6908


In [ ]:
#Combining other text features into df with rating_extreme
rating_features = df[["rating", "rating_extreme"]].values

numeric_features = np.hstack([text_numeric, rating_features])

#viewing head of dataframe with numeric features & rating extreme
numeric_feature_names += ["rating", "rating_extreme"]
pd.DataFrame(numeric_features, columns=numeric_feature_names).head()

,word_count,lexical_diversity,sentiment_extreme,rating,rating_extreme
0,8.0,0.875000,0.9517,5.0,2.0
1,8.0,1.000000,0.8910,5.0,2.0
2,7.0,0.857143,0.7906,5.0,2.0
3,6.0,1.000000,0.4404,1.0,2.0
4,7.0,0.857143,0.6908,5.0,2.0


In [ ]:
#Scaling numeric features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(numeric_features)

numeric_sparse = csr_matrix(numeric_scaled)

#viewing scaled
numeric_scaled.mean(axis=0)
numeric_scaled.std(axis=0)

array([1., 1., 1., 1., 1.])

###TF-IDF
TF: term frequency (how often a word appears in one review)

IDF: inverse document frequency (how rare a word is throughout all reviews)

A word gets a high TF * IDF if it appears often in one review, but not often in other reviews

Fake reviews are more likely to have high TF-IDF values on repetitive, generic phrases.

In [ ]:
#TF-IDF = TF * IDF (term frequency * inverse document frequency)
from sklearn.feature_extraction.text import TfidfVectorizer

word_tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

X_word = word_tfidf.fit_transform(df["text"])

#view top 20 words
X_word.shape
word_tfidf.get_feature_names_out()[:20]

array(['aa', 'aa battery', 'aaa', 'aaa battery', 'aaron', 'ab',
       'abandoned', 'abbey', 'abby', 'abc', 'abducted', 'abigail',
       'ability', 'ability hold', 'ability make', 'able', 'able access',
       'able adjust', 'able amazon', 'able assemble'], dtype=object)

#Character n grams:

* collects windows of 3-5 chars across the review text and converts the groupings of chars into TF*IDF values

Fake reviews usually have high values on repetitive character patterns

In [ ]:
char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=5
)

X_char = char_tfidf.fit_transform(df["text"])

#view top 20 chars
X_char.shape
char_tfidf.get_feature_names_out()[:20]

array([' aa', ' aa ', ' aa b', ' aaa', ' aaa ', ' aar', ' aaro', ' ab',
       ' ab ', ' aba', ' aban', ' abb', ' abbe', ' abby', ' abc', ' abc ',
       ' abd', ' abdu', ' abe', ' abi'], dtype=object)

In [ ]:
from scipy.sparse import hstack

X = hstack([
    X_word,
    X_char,
    numeric_sparse,
    category_ohe
])

y = df["label"].values

num_word = X_word.shape[1]
num_char = X_char.shape[1]
num_numeric = numeric_sparse.shape[1]

numeric_start = num_word + num_char
numeric_end = numeric_start + num_numeric

numeric_part = X[:, numeric_start:numeric_end].toarray()

raw_numeric_df = pd.DataFrame(numeric_features, columns=numeric_feature_names)
raw_numeric_df.head(10)

pd.DataFrame(
    numeric_part[:5],
    columns=[
        "word_count",
        #"sentence_count",
        #"exclamation_count",
        #"uppercase_ratio",
        "lexical_diversity",
        "sentiment_extreme",
        "rating",
        "rating_extreme"
    ]
)


,word_count,lexical_diversity,sentiment_extreme,rating,rating_extreme
0,-0.713546,-0.038527,0.817881,0.649637,0.658058
1,-0.713546,0.930083,0.604069,0.649637,0.658058
2,-0.743765,-0.176900,0.250416,0.649637,0.658058
3,-0.773983,0.930083,-0.983143,-2.845787,0.658058
4,-0.743765,-0.176900,-0.101124,0.649637,0.658058


In [ ]:
df["text"].head(10).tolist()

['love well made sturdy comfortable love itvery pretty',
 'love great upgrade original ive mine couple year',
 'pillow saved back love look feel pillow',
 'missing information use great product price',
 'nice set good quality set two month',
 'wanted different flavor',
 'perfect touch thing wish little space',
 'done fit well look great love smoothness edge extra',
 'great big number easy read thing didnt like size',
 'son love comforter well made also baby']